# Tree model — **pyfunc flavor** → CPU-SMALL serving endpoint

Trains a RandomForest, logs it as a **custom `mlflow.pyfunc` model** (returns the
positive-class probability in Model Serving tabular format), registers to Unity
Catalog, and deploys a **CPU / SMALL** serving endpoint on the shm-skunkworks FEVM.

Dependencies are pinned to the exact training-time versions so the serving
environment reproduces training.

In [ ]:
# Serverless job compute is bare — install what we need, then restart Python.
%pip install -q mlflow scikit-learn pandas numpy
dbutils.library.restartPython()

In [ ]:
CATALOG = "shm_catalog"
SCHEMA = "shared"
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.tree_pyfunc_model"
ENDPOINT_NAME = "shm_tree_pyfunc_endpoint"

In [ ]:
import mlflow, numpy as np, pandas as pd, sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

mlflow.set_registry_uri("databricks-uc")

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
clf.fit(X_train, y_train)
print("train acc:", clf.score(X_train, y_train), " test acc:", clf.score(X_test, y_test))

In [ ]:
class TreeProbaModel(mlflow.pyfunc.PythonModel):
    """Custom pyfunc: returns the positive-class probability in Databricks
    Model Serving tabular format -> {"predictions": [...]}."""

    def __init__(self, model):
        self.model = model

    def predict(self, context, model_input: pd.DataFrame):
        proba = self.model.predict_proba(model_input)[:, 1]
        return {"predictions": proba.tolist()}

In [ ]:
from mlflow.models import infer_signature

wrapped = TreeProbaModel(clf)
example = X_test.head(3)
signature = infer_signature(example, wrapped.predict(None, example))

# Pin exact training-time versions so the serving container matches training.
pip_requirements = [
    f"mlflow=={mlflow.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"pandas=={pd.__version__}",
    f"numpy=={np.__version__}",
]
print("pip_requirements:", pip_requirements)

with mlflow.start_run(run_name="tree_pyfunc"):
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=wrapped,
        signature=signature,
        input_example=example,
        registered_model_name=UC_MODEL_NAME,
        pip_requirements=pip_requirements,
    )
print("registered version:", info.registered_model_version)

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    ServedModelInput,
    EndpointCoreConfigInput,
    ServedModelInputWorkloadSize,
    ServedModelInputWorkloadType,
)

w = WorkspaceClient()
served = ServedModelInput(
    model_name=UC_MODEL_NAME,
    model_version=info.registered_model_version,
    workload_type=ServedModelInputWorkloadType.CPU,     # CPU
    workload_size=ServedModelInputWorkloadSize.SMALL,   # SMALL
    scale_to_zero_enabled=True,
)

try:
    w.serving_endpoints.update_config(name=ENDPOINT_NAME, served_models=[served]).result()
    print(f"✅ updated endpoint {ENDPOINT_NAME}")
except Exception:
    w.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(served_models=[served]),
    ).result()
    print(f"✅ created endpoint {ENDPOINT_NAME}")

In [ ]:
from databricks.sdk.service.serving import DataframeSplitInput

resp = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_split=DataframeSplitInput(
        columns=list(X_test.columns),
        data=X_test.head(3).values.tolist(),
    ),
)
print("predictions:", resp.predictions)